# 🔢 Notebook 2: Math Reasoning + Code-Mixed QA

**Indic Language Benchmark Suite — Day 2**

This notebook:
1. Loads IndicMMLU-Pro (math/STEM subsets) for Hindi, Bengali, Tamil
2. Loads 50 hand-crafted Hinglish code-mixed QA samples
3. Runs inference on all three models for both tasks
4. Computes Accuracy (math) and EM/F1/BLEU (code-mixed)
5. Builds a full comparison table across all 3 tasks
6. Generates visualization charts

## 1. Setup

In [ ]:
%%capture
!pip install -q transformers>=4.40.0 torch accelerate bitsandbytes
!pip install -q datasets sentencepiece protobuf evaluate
!pip install -q rouge-score nltk pandas tabulate tqdm jsonlines
!pip install -q matplotlib seaborn huggingface_hub

In [ ]:
import os
import sys
import torch
import pandas as pd

# Add project root to path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.getcwd())

from src.data_loader import load_indicmmlu, load_code_mixed_qa
from src.model_loader import load_model_and_tokenizer, unload_model
from src.inference import run_inference
from src.metrics import evaluate_results, format_metrics_report
from src.results import build_leaderboard, generate_all_charts, load_all_results

# HuggingFace login (for gated models)
from huggingface_hub import notebook_login
notebook_login()

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 2. Load Datasets

In [ ]:
# ─── Task 2: IndicMMLU-Pro (Math/STEM) ───
MAX_MATH_SAMPLES = 100  # Set to 200 for final run

math_samples = load_indicmmlu(
    languages=["hindi", "bengali", "tamil"],
    max_samples_per_lang=MAX_MATH_SAMPLES,
    stem_only=True,
)

print(f"\nMath samples loaded: {len(math_samples)}")
print(pd.DataFrame(math_samples)['language'].value_counts())

In [ ]:
# ─── Task 3: Code-Mixed QA (Hinglish) ───
codemixed_samples = load_code_mixed_qa("data/code_mixed_qa.json")

print(f"\nCode-mixed samples loaded: {len(codemixed_samples)}")
# Show a few examples
for s in codemixed_samples[:3]:
    print(f"  Q: {s['question']}")
    print(f"  A: {s['answer']}")
    print()

In [ ]:
# Inspect a math sample
if math_samples:
    sample = math_samples[0]
    print("Sample math question:")
    print(f"  Q: {sample['question'][:200]}")
    print(f"  Options: {sample.get('options', 'N/A')}")
    print(f"  Answer: {sample['answer']}")
    print(f"  Language: {sample['language']}")

## 3. Run Inference — Math Reasoning

In [ ]:
RESULTS_DIR = "results/raw"
os.makedirs(RESULTS_DIR, exist_ok=True)

MODELS_TO_EVAL = ["sarvam-2b", "gemma-2b", "llama-3.2-1b"]
math_results_all = {}

for model_key in MODELS_TO_EVAL:
    print(f"\n{'='*60}")
    print(f"📐 Evaluating {model_key} on Math Reasoning")
    print(f"{'='*60}")
    
    model, tokenizer, config = load_model_and_tokenizer(model_key, quantize=True)
    
    results = run_inference(
        model=model,
        tokenizer=tokenizer,
        samples=math_samples,
        task="math_reasoning",
        model_name=config.name,
        max_new_tokens=16,  # Just need the option letter
        use_fewshot=True,
        save_path=f"{RESULTS_DIR}/{model_key}_math_reasoning.jsonl",
    )
    
    math_results_all[model_key] = results
    
    # Evaluate
    metrics = evaluate_results(results, "math_reasoning")
    print(format_metrics_report(metrics, config.name))
    
    unload_model(model, tokenizer)

## 4. Run Inference — Code-Mixed QA

In [ ]:
codemixed_results_all = {}

for model_key in MODELS_TO_EVAL:
    print(f"\n{'='*60}")
    print(f"🗣️  Evaluating {model_key} on Code-Mixed QA")
    print(f"{'='*60}")
    
    model, tokenizer, config = load_model_and_tokenizer(model_key, quantize=True)
    
    results = run_inference(
        model=model,
        tokenizer=tokenizer,
        samples=codemixed_samples,
        task="code_mixed_qa",
        model_name=config.name,
        max_new_tokens=48,
        use_fewshot=False,  # Test raw code-mixing ability
        save_path=f"{RESULTS_DIR}/{model_key}_code_mixed_qa.jsonl",
    )
    
    codemixed_results_all[model_key] = results
    
    # Evaluate
    metrics = evaluate_results(results, "code_mixed_qa")
    print(format_metrics_report(metrics, config.name))
    
    unload_model(model, tokenizer)

## 5. Full Comparison Table

In [ ]:
# Load all results (including Day 1's reading comprehension)
all_results = load_all_results(RESULTS_DIR)

# Build leaderboard
leaderboard = build_leaderboard(all_results)

print("\n" + "=" * 80)
print("🏆 FULL BENCHMARK LEADERBOARD")
print("=" * 80)
print(leaderboard.to_markdown(index=False))

In [ ]:
# Summary view
from src.results import build_summary_table

summary = build_summary_table(leaderboard)
print("\n📊 SUMMARY (Primary Metric per Task):")
print(summary.to_markdown())

## 6. Generate Visualizations

In [ ]:
# Generate all charts
generate_all_charts(leaderboard, output_dir="results/figures")

# Display charts inline
import matplotlib.pyplot as plt
from IPython.display import Image, display

for chart_file in ["model_comparison.png", "language_heatmap.png", "radar_chart.png"]:
    path = f"results/figures/{chart_file}"
    if os.path.exists(path):
        print(f"\n📊 {chart_file}:")
        display(Image(filename=path))

## 7. Qualitative Analysis

In [ ]:
# Spot-check code-mixed predictions
print("\n🗣️  CODE-MIXED QA — Sample Predictions")
print("-" * 70)

for model_key, results in codemixed_results_all.items():
    print(f"\n🤖 {model_key}:")
    for r in results[:5]:
        print(f"  Q: {r['question']}")
        print(f"  Gold: {r['gold_answer']}")
        print(f"  Pred: {r['prediction'][:100]}")
        print()

In [ ]:
# Spot-check math predictions
print("\n📐 MATH REASONING — Sample Predictions")
print("-" * 70)

for model_key, results in math_results_all.items():
    print(f"\n🤖 {model_key}:")
    for r in results[:5]:
        print(f"  Q: {r['question'][:100]}...")
        print(f"  Gold: {r['gold_answer']}")
        print(f"  Pred: {r['prediction'][:50]}")
        print()

In [ ]:
# Save the leaderboard as markdown
from src.results import save_leaderboard_markdown
save_leaderboard_markdown(leaderboard, "results/leaderboard.md")

print("\n✅ Day 2 complete!")
print("📂 All results saved in results/")
print("📊 Charts saved in results/figures/")
print("📝 Leaderboard saved in results/leaderboard.md")
print("\n→ Proceed to Notebook 3 for publication and analysis.")